# 문제 1 — 벡터 연산 모듈 (내적·외적·정사영·rank)

로봇의 좌표 변환은 결국 **벡터 연산**의 조합입니다. 이 노트북에서는
내적·사이각·정규화·정사영·반대칭행렬(외적)·평면 법선·rank 를
`np.linalg` 없이 직접 구현하고, 각각을 검증합니다.

완성한 함수는 `src/vectors.py` 에 채워 넣어 문제 2 이후에서 재사용합니다.

## 이 노트북에서 해야 할 일

| # | 할 일 | 구현할 함수 |
|---|---|---|
| 1-1 | 내적과 사이각을 구하고 **손계산 값과 일치**하는지 검증 | `dot`, `norm`, `angle_between` |
| 1-2 | 정규화 함수를 만들고 **영벡터를 넣으면 어떻게 되는지 직접 실행해 기록**한 뒤 처리 방식을 정해 구현 | `normalize` |
| 1-3 | 정사영을 구현하고 ① 남는 성분이 수직인지 ② 두 성분의 합이 원래 벡터인지 검증 | `project`, `reject` |
| 1-4 | 외적을 **반대칭행렬 곱**으로 구현하고 `np.cross` 와 비교, 반대칭성 검증 | `skew`, `cross` |
| 1-5 | 세 점이 만드는 평면의 **단위 법선** | `plane_normal` |
| 1-6 | (1,0,1), (0,1,1), (1,1,2) 의 rank 를 구하고 **왜 3 이 아닌지** 설명, 행렬식과 일관성 확인 | `row_echelon`, `rank`, `det` |

> **규약**
> - 난수는 `np.random.default_rng(42)` 로 고정합니다.
> - 수치 비교는 부동소수점 오차를 고려해 `np.allclose` / `np.isclose` 로 합니다.
> - `np.linalg` 는 **검산용으로만** 쓰고, 쓸 때마다 주석으로 검산임을 밝힙니다.
> - 각 문항은 **(1) 설명 마크다운 → (2) 코드 → (3) 검증** 순서를 지킵니다.

In [1]:
import sys
from pathlib import Path

import numpy as np

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.vectors import (angle_between, cross, det, dot, norm, normalize,
                         plane_normal, project, rank, reject, row_echelon, skew)

rng = np.random.default_rng(42)          # 시드 고정
np.set_printoptions(precision=6, suppress=True)


def check(label, condition):
    """검증 셀에서 쓰는 통과/실패 출력 헬퍼. (그대로 쓰면 됩니다)"""
    tag = "PASS" if condition else "FAIL"
    print("[" + tag + "] " + label)
    return bool(condition)


print("NumPy", np.__version__)

NumPy 2.2.6


## 1-1. 내적과 사이각

내적의 정의는 두 가지이며 서로 같습니다.

$$\mathbf{a}\cdot\mathbf{b}=\sum_i a_i b_i = |\mathbf{a}||\mathbf{b}|\cos\theta$$

두 번째 식을 $\theta$ 에 대해 풀면 사이각이 나옵니다.

검증하기 쉽도록 **손으로 계산되는 값**을 고릅니다.
$\mathbf{a}=(3,4,0)$, $\mathbf{b}=(4,3,0)$ 이면 $|\mathbf{a}|=|\mathbf{b}|=5$ 이므로
내적과 $\cos\theta$, 사이각을 종이에서 먼저 구한 뒤 코드 결과와 비교하세요.

**할 일** — `src/vectors.py` 의 `dot`, `norm`, `angle_between` 을 구현하고
아래 셀에서 손계산 값과 나란히 출력합니다.

In [2]:
a = np.array([3.0, 4.0, 0.0])
b = np.array([4.0, 3.0, 0.0])

d = dot(a,b)
theta_deg = round(angle_between(a,b),2)

hand_dot = 3*4 + 4*3 + 0*0

hand_norm_a = np.sqrt(3**2 + 4**2)
hand_norm_b = np.sqrt(4**2 + 3**2)

hand_cos = hand_dot / (hand_norm_a * hand_norm_b)

hand_deg = round(np.degrees(np.arccos(hand_cos)),2)

# 나란히 출력
print("=== 함수 계산 ===")
print("dot =", d)
print("angle =", theta_deg, "degrees")

print("\n=== 손계산 ===")
print("dot =", hand_dot)
print("cos(theta) =", hand_cos)
print("angle =", hand_deg, "degrees")

print("\n=== 비교 ===")
print("dot 일치:", np.isclose(d, hand_dot))
print("angle 일치:", np.isclose(theta_deg, hand_deg))

# TODO: dot / norm / angle_between 을 호출해 아래 값을 구하고 출력하세요.
#   d          = ...
#   theta_deg  = ...
# TODO: 손으로 계산한 값(hand_dot, hand_cos, hand_deg)도 함께 출력해 비교하세요.

=== 함수 계산 ===
dot = 24.0
angle = 16.26 degrees

=== 손계산 ===
dot = 24
cos(theta) = 0.96
angle = 16.26 degrees

=== 비교 ===
dot 일치: True
angle 일치: True


In [3]:
# --- 검증 ---

# 내적이 손계산 값과 일치하는가
check(
    "내적이 손계산 값과 일치",
    np.isclose(d, hand_dot)
)

# 사이각이 손계산 값과 일치하는가
check(
    "사이각이 손계산 값과 일치",
    np.isclose(theta_deg, hand_deg)
)

# 검산용
check(
    "np.dot 검산 결과와 일치",
    np.isclose(d, np.dot(a, b))
)

# 수직인 두 벡터의 사이각이 90도인가
v1 = np.array([1.0, 0.0, 0.0])
v2 = np.array([0.0, 1.0, 0.0])

check(
    "수직인 두 벡터의 사이각이 90도",
    np.isclose(angle_between(v1, v2), 90.0)
)

# 같은 벡터끼리의 사이각이 0도인가
v3 = np.array([1.0, 2.0, 3.0])

check(
    "같은 벡터끼리의 사이각이 0도",
    np.isclose(angle_between(v3, v3), 0.0)
)

[PASS] 내적이 손계산 값과 일치
[PASS] 사이각이 손계산 값과 일치
[PASS] np.dot 검산 결과와 일치
[PASS] 수직인 두 벡터의 사이각이 90도
[PASS] 같은 벡터끼리의 사이각이 0도


True

## 1-2. 정규화와 영벡터 — 무슨 일이 일어나는가

정규화는 $\hat{\mathbf{v}} = \mathbf{v}/|\mathbf{v}|$ 입니다.

**할 일**

1. 먼저 **아무 보호 장치 없이** 영벡터를 길이로 나눠 보고, 실제로 무엇이 출력되는지
   (경고 메시지 포함) 그대로 기록하세요.
   경고를 죽이고 관찰하려면 `with np.errstate(invalid="ignore", divide="ignore"):` 를 쓰면 됩니다.
2. 그 결과가 **왜 위험한지** 생각해 보세요. 이후 연산에 어떻게 전파되는지,
   `assert` 나 `==` 비교로 잡히는지 직접 확인해 보면 답이 보입니다.
3. 어떻게 처리할지 **직접 정하고**(예: 예외를 던진다 / 영벡터를 그대로 돌려준다 /
   특정 축을 돌려준다 …) `src/vectors.py` 의 `normalize` 에 구현하세요.
4. 아래 마크다운에 **선택한 방식과 근거**를 적으세요. 검증 셀도 그 방식에 맞춰 작성합니다.

### 선택한 처리 방식과 근거

- 관찰한 결과: `___`
- 선택한 처리: `___`
- 근거: `___`

In [4]:
zero = np.array([0.0, 0.0, 0.0])
unsafe = zero / np.linalg.norm(zero)

# TODO: (1) 보호 없이 나눴을 때 무슨 값이 나오는지 관찰해 출력하세요.
print("=== (1) 보호 없이 정규화 ===")
print("결과:", unsafe)
print("값 확인:", unsafe[0])
print("isnan:", np.isnan(unsafe))
print("isinf:", np.isinf(unsafe))


# TODO: (2) 그 값이 이후 비교/전파에서 어떻게 동작하는지 확인해 출력하세요.

print("\n=== (2) 이후 계산에서의 동작 ===")

print("unsafe == 0:", unsafe == 0)
print("np.allclose(unsafe, 0):", np.allclose(unsafe, 0))
print("norm(unsafe):", np.linalg.norm(unsafe))

propagated = unsafe + np.array([1.0, 2.0, 3.0])
print("unsafe + [1,2,3]:", propagated)

print("np.isnan(propagated):", np.isnan(propagated))


# TODO: (3) 구현한 normalize 로 정상 벡터와 영벡터를 각각 처리해 출력하세요.
print("\n=== (3) 구현한 normalize ===")

normal = np.array([3.0, 4.0, 0.0])

print("정상 벡터:")
print("입력:", normal)
print("결과:", normalize(normal))

print("\n영벡터:")
try:
    result = normalize(zero)
    print("결과:", result)
except ValueError as e:
    print("예외:", e)

=== (1) 보호 없이 정규화 ===
결과: [nan nan nan]
값 확인: nan
isnan: [ True  True  True]
isinf: [False False False]

=== (2) 이후 계산에서의 동작 ===
unsafe == 0: [False False False]
np.allclose(unsafe, 0): False
norm(unsafe): nan
unsafe + [1,2,3]: [nan nan nan]
np.isnan(propagated): [ True  True  True]

=== (3) 구현한 normalize ===
정상 벡터:
입력: [3. 4. 0.]
결과: [0.6 0.8 0. ]

영벡터:
예외: 영벡터는 정규화할 수 없습니다.


/tmp/ipykernel_18004/3752344438.py:2: RuntimeWarning: invalid value encountered in divide
  unsafe = zero / np.linalg.norm(zero)


In [5]:
# --- 검증 ---

# 보호 없이 나눴을 때 관찰한 현상이 실제로 재현되는가
check(
    "보호 없이 영벡터를 나누면 nan이 발생한다",
    np.all(np.isnan(unsafe))
)

# 정규화된 벡터의 길이가 1인가
normal_unit = normalize(normal)

check(
    "정규화된 벡터의 길이가 1이다",
    np.isclose(np.linalg.norm(normal_unit), 1.0)
)

# 정규화가 방향을 바꾸지 않는가 (원본과의 사이각이 0)
check(
    "정규화 전후 사이각이 0도이다",
    np.isclose(angle_between(normal, normal_unit), 0.0)
)

# 영벡터 입력에서 내가 정한 처리 방식대로 동작하는가
try:
    normalize(zero)
    zero_handled = False
except ValueError:
    zero_handled = True

check(
    "영벡터 입력 시 ValueError가 발생한다",
    zero_handled
)

# 무작위 벡터도 정규화 후 길이가 1인가
rng = np.random.default_rng(42)

random_vectors = rng.normal(size=(10, 3))

random_results = [
    normalize(v)
    for v in random_vectors
]

random_norms = [
    np.linalg.norm(v)
    for v in random_results
]

check(
    "무작위 벡터도 정규화 후 길이가 1이다",
    np.allclose(random_norms, 1.0)
)

[PASS] 보호 없이 영벡터를 나누면 nan이 발생한다
[PASS] 정규화된 벡터의 길이가 1이다
[PASS] 정규화 전후 사이각이 0도이다
[PASS] 영벡터 입력 시 ValueError가 발생한다
[PASS] 무작위 벡터도 정규화 후 길이가 1이다


True

## 1-3. 정사영 — 수직성과 합 복원

$\mathbf{a}$ 를 $\mathbf{b}$ 방향으로 정사영한 성분은

$$\mathrm{proj}_{\mathbf{b}}(\mathbf{a})=\frac{\mathbf{a}\cdot\mathbf{b}}{\mathbf{b}\cdot\mathbf{b}}\mathbf{b}$$

이고, 남는 성분(reject)은 $\mathbf{a}-\mathrm{proj}_{\mathbf{b}}(\mathbf{a})$ 입니다.
분모가 $|\mathbf{b}|^2$ 이므로 $\mathbf{b}$ 를 미리 정규화할 필요가 없습니다.

**검증해야 할 두 가지**

1. **수직성**: (남는 성분) $\cdot$ $\mathbf{b} = 0$
2. **합 복원**: $\mathrm{proj} + \mathrm{rej} = \mathbf{a}$

In [6]:
a = np.array([2.0, 3.0, 4.0])
b = np.array([1.0, 0.0, 1.0])

p =project (a,b)
r = reject(a,b)

c=np.dot(a,b) / np.dot(b,b)

print("a = ", a)
print("b = ", b)
print("계수 (a*b)/(b*b) = " ,c)
print("project(a, b) =", p)
print("reject(a, b) =", r)

# TODO: project / reject 를 호출하고, 계수 (a·b)/(b·b) 와 함께 결과를 출력하세요.

a =  [2. 3. 4.]
b =  [1. 0. 1.]
계수 (a*b)/(b*b) =  3.0
project(a, b) = [3. 0. 3.]
reject(a, b) = [-1.  3.  1.]


In [7]:
# --- 검증 ---

# ① 남는 성분이 b와 수직인가 (rej · b = 0)
check(
    "① reject와 b가 수직이다",
    np.isclose(np.dot(r, b), 0.0)
)

# ② proj + rej = a 인가
check(
    "② proj + reject = a",
    np.allclose(p + r, a)
)

# proj가 b와 평행한가 (외적이 0)
check(
    "proj와 b가 평행하다",
    np.allclose(np.cross(p, b), 0.0)
)

# 피타고라스: |a|² = |proj|² + |rej|²
check(
    "피타고라스 관계가 성립한다",
    np.isclose(
        np.dot(a, a),
        np.dot(p, p) + np.dot(r, r)
    )
)

# 무작위 100쌍에서도 ①, ②가 모두 성립하는가
rng = np.random.default_rng(42)

all_valid = True

for _ in range(100):
    random_a = rng.normal(size=3)
    random_b = rng.normal(size=3)

    random_p = project(random_a, random_b)
    random_r = reject(random_a, random_b)

    # ① reject · b = 0
    perpendicular = np.isclose(
        np.dot(random_r, random_b),
        0.0
    )

    # ② project + reject = a
    reconstruction = np.allclose(
        random_p + random_r,
        random_a
    )

    if not (perpendicular and reconstruction):
        all_valid = False
        break

check(
    "무작위 100쌍에서 ①②가 모두 성립한다",
    all_valid
)

[PASS] ① reject와 b가 수직이다
[PASS] ② proj + reject = a
[PASS] proj와 b가 평행하다
[PASS] 피타고라스 관계가 성립한다
[PASS] 무작위 100쌍에서 ①②가 모두 성립한다


True

## 1-4. 외적을 반대칭행렬 곱으로 — `skew(a)`

외적은 행렬 곱으로 쓸 수 있습니다.

$$\mathbf{a}\times\mathbf{b} = [\mathbf{a}]_\times \mathbf{b},\qquad
[\mathbf{a}]_\times=\begin{bmatrix}0&-a_3&a_2\a_3&0&-a_1\-a_2&a_1&0\end{bmatrix}$$

이 형태가 중요한 이유는 **로드리게스 공식(문제 2)과 각속도 → 회전 미분**이
전부 $[\boldsymbol{\omega}]_\times$ 로 표현되기 때문입니다.
반대칭(skew-symmetric)이란 $M^{\mathsf{T}} = -M$ 을 뜻합니다.
여기서 따라오는 성질이 하나 더 있는데, 직접 출력해서 확인해 보세요.

In [8]:
a = np.array([1.0, 2.0, 3.0])
b = np.array([4.0, 5.0, 6.0])

# TODO: skew(a) 를 출력하고, skew(a) @ b 와 np.cross(a, b)(# 검산용)를 비교하세요.
aKew = skew(a)
bsKew = skew(a) @ b
print( "skew(a) = \n", aKew)
print("\n")
print(" skew(a) @ b = \n", bsKew)
print("\n")
print(" 검산 : \n",np.cross(a,b))
print("\n")
# TODO: skew(a).T 와 -skew(a) 를 나란히 출력해 반대칭성을 눈으로 확인하세요.

print("skew(a).T = \n",aKew.T)
print("\n")
print("-skew(a) = \n",-aKew)
print("\n")

skew(a) = 
 [[ 0. -3.  2.]
 [ 3.  0. -1.]
 [-2.  1.  0.]]


 skew(a) @ b = 
 [-3.  6. -3.]


 검산 : 
 [-3.  6. -3.]


skew(a).T = 
 [[ 0.  3. -2.]
 [-3.  0.  1.]
 [ 2. -1.  0.]]


-skew(a) = 
 [[-0.  3. -2.]
 [-3. -0.  1.]
 [ 2. -1. -0.]]




In [9]:
# --- 검증 ---

S = skew(a)

# 1. skew(a) @ b == np.cross(a, b)
check(
    "skew(a) @ b == np.cross(a, b)",
    np.allclose(S @ b, np.cross(a, b))
)

# 2. skew(a)가 반대칭인가
check(
    "skew(a)가 반대칭이다",
    np.allclose(S.T, -S)
)

# 3. 반대칭에서 따라오는 대각성분 성질
# 반대칭행렬의 대각성분은 모두 0이어야 한다.
check(
    "skew(a)의 대각성분이 모두 0이다",
    np.allclose(np.diag(S), 0.0)
)

# 4. skew(a) @ a == 0
# 자기 자신과의 외적은 0
check(
    "skew(a) @ a == 0",
    np.allclose(S @ a, 0.0)
)

# 5. 반교환성: a × b == -(b × a)
check(
    "반교환성이 성립한다",
    np.allclose(np.cross(a, b), -np.cross(b, a))
)

# 6. 무작위 200쌍에서 skew 곱 == np.cross
rng = np.random.default_rng(42)

all_valid = True

for _ in range(200):
    random_a = rng.normal(size=3)
    random_b = rng.normal(size=3)

    if not np.allclose(
        skew(random_a) @ random_b,
        np.cross(random_a, random_b)
    ):
        all_valid = False
        break

check(
    "무작위 200쌍에서 skew 곱 == np.cross",
    all_valid
)

[PASS] skew(a) @ b == np.cross(a, b)
[PASS] skew(a)가 반대칭이다
[PASS] skew(a)의 대각성분이 모두 0이다
[PASS] skew(a) @ a == 0
[PASS] 반교환성이 성립한다
[PASS] 무작위 200쌍에서 skew 곱 == np.cross


True

## 1-5. 세 점이 만드는 평면의 단위 법선

세 점 $P_1,P_2,P_3$ 이 주어지면 두 모서리 벡터
$\mathbf{u}=P_2-P_1$, $\mathbf{v}=P_3-P_1$ 의 외적이 평면에 수직입니다.
이를 정규화하면 단위 법선입니다.

세 점이 **일직선**이면 어떻게 될지 먼저 생각해 보고, 그 경우를 어떻게 처리할지 정하세요.

In [10]:
P1 = np.array([0.0, 0.0, 0.0])
P2 = np.array([1.0, 0.0, 0.0])
P3 = np.array([0.0, 1.0, 0.0])

u = P1 - P2
v = P3 - P1
expect = np.array([0.0,0.0,1.0])

# TODO: xy 평면 위 세 점의 단위 법선을 구해 출력하고, 기대값과 비교하세요.

print("--- xy 평면 위 세점의 단위 법선 ---")
n = plane_normal(P1,P2,P3)
print("단위 법선:", n)
print("기대값:", expect)


# TODO: 기울어진 평면(예: (1,0,0), (0,1,0), (0,0,1))에서도 구해 보세요.


print("--- 기울어진 평면 ---")
print("기울어진 평면의 단위 법선:", n)
print("법선의 크기:", np.linalg.norm(n))



# TODO: 일직선인 세 점을 넣으면 어떻게 되는지 확인해 출력하세요.

P1_line = np.array([0.0, 0.0, 0.0])
P2_line = np.array([1.0, 1.0, 1.0])
P3_line = np.array([2.0, 2.0, 2.0])

print("--- 일직선인 세 점 ---")

try:
    n_line = plane_normal(P1_line, P2_line, P3_line)
    print("단위 법선:", n_line)
except ValueError as e:
    print("오류:", e)

--- xy 평면 위 세점의 단위 법선 ---
단위 법선: [0. 0. 1.]
기대값: [0. 0. 1.]
--- 기울어진 평면 ---
기울어진 평면의 단위 법선: [0. 0. 1.]
법선의 크기: 1.0
--- 일직선인 세 점 ---
오류: 세 점이 일직선이므로 법선을 정의할 수 없습니다.


In [11]:
# --- 검증 ---

# 1. 법선의 길이가 1인가
check(
    "법선의 길이가 1이다",
    np.isclose(np.linalg.norm(n), 1.0)
)

# 2. xy 평면의 법선이 z축과 일치하는가
z_axis = np.array([0.0, 0.0, 1.0])

check(
    "xy 평면의 법선이 z축과 일치한다",
    np.allclose(n, z_axis)
)

# 3. 법선이 두 모서리 벡터 모두와 수직인가
u = P2 - P1
v = P3 - P1

check(
    "법선이 첫 번째 모서리 벡터와 수직이다",
    np.isclose(np.dot(n, u), 0.0)
)

check(
    "법선이 두 번째 모서리 벡터와 수직이다",
    np.isclose(np.dot(n, v), 0.0)
)

# 4. 기울어진 평면의 법선이 기대값과 일치하는가
check(
    "기울어진 평면의 법선이 기대값과 일치한다",
    np.allclose(n, expect)
)

# 5. 일직선 입력에서 내가 정한 처리 방식대로 동작하는가
try:
    plane_normal(P1_line, P2_line, P3_line)
    line_handled = False
except ValueError:
    line_handled = True

check(
    "일직선 입력에서 ValueError가 발생한다",
    line_handled
)

[PASS] 법선의 길이가 1이다
[PASS] xy 평면의 법선이 z축과 일치한다
[PASS] 법선이 첫 번째 모서리 벡터와 수직이다
[PASS] 법선이 두 번째 모서리 벡터와 수직이다
[PASS] 기울어진 평면의 법선이 기대값과 일치한다
[PASS] 일직선 입력에서 ValueError가 발생한다


True

## 1-6. rank 와 행렬식 — 왜 3 이 아닌가

세 벡터 $(1,0,1)$, $(0,1,1)$, $(1,1,2)$ 를 행으로 쌓은 행렬의 rank 를 구합니다.
rank 는 **선형독립인 행(또는 열)의 개수**이고, 행 사다리꼴로 만들었을 때
살아남는 피벗의 개수와 같습니다.

코드를 돌리기 **전에** 세 벡터를 눈으로 보고 서로 어떤 관계인지 찾아보세요.
그 관계가 곧 "왜 3 이 아닌가" 의 답입니다.
정사각 행렬에서 rank 와 행렬식은 서로 일관돼야 한다는 점도 확인합니다.

**할 일** — `row_echelon`, `rank`, `det` 를 구현하고 아래를 채우세요.

### rank 가 3 이 아닌 이유

- 발견한 선형종속 관계: `___`
- 세 벡터가 span 하는 공간: `___`
- rank 와 행렬식이 일관되는 이유: `___`

In [12]:
M = np.array([[1.0, 0.0, 1.0],
              [0.0, 1.0, 1.0],
              [1.0, 1.0, 2.0]])

# TODO: row_echelon 으로 사다리꼴과 피벗 열을 출력하세요.
print("row_echeloon")
E = row_echelon(M)
print(E)
pivot_col = []

for i in range(E.shape[0]):
    nozero = np.where(~np.isclose(E[i],0.0))[0]

    if len(nozero)> 0:
     pivot_col.append(nozero[0])

print("피벗 열 : ",pivot_col)


# TODO: 직접 구현한 rank / det 를 np.linalg.matrix_rank / np.linalg.det (# 검산용) 와 비교하세요.

Xrank = rank(M)
Xdet = det(M)
print("\n=== rank / det ===")

print("직접 구현 rank:", Xrank)
print("np.linalg.matrix_rank:", np.linalg.matrix_rank(M))  # 검산용

print("직접 구현 det:", Xdet)
print("np.linalg.det:", np.linalg.det(M))  # 검산용

# TODO: 세 벡터 사이의 선형종속 관계를 코드로 확인해 출력하세요.
v1 = M[0]
v2 = M[1]
v3 = M[2]

print("\n=== 선형종속 관계 ===")

print("v1 =", v1)
print("v2 =", v2)
print("v3 =", v3)

print("v1 + v2 =", v1 + v2)

check(
    "v3 = v1 + v2",
    np.allclose(v3, v1 + v2)
)


#       (스칼라 삼중곱 dot(v1, cross(v2, v3)) 도 같이 보면 좋습니다)

triple = dot(v1, cross(v2, v3))

print("스칼라 삼중곱:", triple)

check(
    "스칼라 삼중곱이 0이다",
    np.isclose(triple, 0.0)
)

row_echeloon
[[1. 0. 1.]
 [0. 1. 1.]
 [0. 0. 0.]]
피벗 열 :  [np.int64(0), np.int64(1)]

=== rank / det ===
직접 구현 rank: 2
np.linalg.matrix_rank: 2
직접 구현 det: 0.0
np.linalg.det: 0.0

=== 선형종속 관계 ===
v1 = [1. 0. 1.]
v2 = [0. 1. 1.]
v3 = [1. 1. 2.]
v1 + v2 = [1. 1. 2.]
[PASS] v3 = v1 + v2
스칼라 삼중곱: 0.0
[PASS] 스칼라 삼중곱이 0이다


True

In [13]:
# --- 검증 ---

# 1. rank가 3이 아닌 값인가
check(
    "rank가 3이 아니다",
    Xrank != 3
)

# 2. 직접 구현 rank == np.linalg.matrix_rank
check(
    "직접 구현 rank == np.linalg.matrix_rank",
    Xrank == np.linalg.matrix_rank(M)
)

# 3. 찾아낸 선형종속 관계가 실제로 성립하는가
check(
    "v3 = v1 + v2 관계가 성립한다",
    np.allclose(v3, v1 + v2)
)

# 4. 행렬식이 0인가
check(
    "행렬식이 0이다",
    np.isclose(Xdet, 0.0)
)

# 직접 구현 det == np.linalg.det
check(
    "직접 구현 det == np.linalg.det",
    np.isclose(Xdet, np.linalg.det(M))
)

# 5. rank < 3 과 det == 0 이 서로 일관되는가
check(
    "rank < 3 과 det == 0 이 일관된다",
    (Xrank < 3) == np.isclose(Xdet, 0.0)
)

# 6. 반례: 단위행렬은 rank 3, det 1 인가
I = np.eye(3)

I_rank = rank(I)
I_det = det(I)

check(
    "단위행렬의 rank는 3이다",
    I_rank == 3
)

check(
    "단위행렬의 det는 1이다",
    np.isclose(I_det, 1.0)
)

[PASS] rank가 3이 아니다
[PASS] 직접 구현 rank == np.linalg.matrix_rank
[PASS] v3 = v1 + v2 관계가 성립한다
[PASS] 행렬식이 0이다
[PASS] 직접 구현 det == np.linalg.det
[PASS] rank < 3 과 det == 0 이 일관된다
[PASS] 단위행렬의 rank는 3이다
[PASS] 단위행렬의 det는 1이다


True

## 답안 템플릿 정리

지시문의 답안 템플릿에 맞춰 아래 빈칸을 채워 출력하세요.
값은 위에서 계산한 변수를 그대로 넣고, 설명은 직접 문장으로 씁니다.

In [14]:
summary = """
1. 내적: 24 / 사이각: 16.26 도
   - 손계산과 일치 여부: 일치함

2. 영벡터 정규화 시 결과: 값없음
   - 선택한 처리: valueEroor발생
   - 근거: 0벡터는 크기가 0이므로 0으로 나눌수없음

3. 정사영 검증: 수직성 true / 합 복원 true

4. skew(a) @ b 와 np.cross(a, b) 일치: 일치

5. 세 벡터의 rank: 2
   - 3 이 아닌 이유:  3벡터가 선형 종속
   - 행렬식 값: 0 -> rank 와 일관되는가: 일관됨ㅇㅇ
"""
print(summary)


1. 내적: 24 / 사이각: 16.26 도
   - 손계산과 일치 여부: 일치함

2. 영벡터 정규화 시 결과: 값없음
   - 선택한 처리: valueEroor발생
   - 근거: 0벡터는 크기가 0이므로 0으로 나눌수없음

3. 정사영 검증: 수직성 true / 합 복원 true

4. skew(a) @ b 와 np.cross(a, b) 일치: 일치

5. 세 벡터의 rank: 2
   - 3 이 아닌 이유:  3벡터가 선형 종속
   - 행렬식 값: 0 -> rank 와 일관되는가: 일관됨ㅇㅇ

